# LINGUA AI - MarianMT Sequence-to-Sequence Model Fine-Tuning
### Google Colab GPU-Accelerated Training Notebook

This notebook trains and fine-tunes a **MarianMT Sequence-to-Sequence Transformer** model for **English to Hindi** and **Hindi to English** translation using the OPUS parallel corpus and PyTorch/Hugging Face Transformers.

In [ ]:
# 1. Check GPU availability
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Install required packages
!pip install -q torch transformers datasets sentencepiece sacrebleu accelerate matplotlib pandas scikit-learn

In [ ]:
# 3. Import dependencies
import os
import re
import time
import json
import unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import sacrebleu
from transformers import (
    MarianMTForConditionalGeneration,
    MarianTokenizer,
    get_linear_schedule_with_warmup
)
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

In [ ]:
# 4. Text cleaning and Devanagari-preserving normalization
def clean_sentence(text, is_hindi=False):
    if not isinstance(text, str) or pd.isna(text):
        return ""
    # Unicode normalization (NFC ensures Devanagari combined matras stay intact)
    text = unicodedata.normalize("NFC", text)
    # Strip control characters
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]", "", text)
    # Remove zero-width spaces
    text = text.replace("\u200b", "").replace("\ufeff", "").replace("\u00a0", " ")
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 5. Download OPUS Parallel Dataset
print("Loading OPUS English-Hindi parallel corpus...")
try:
    raw_dataset = load_dataset("cfilt/iitb-english-hindi", split="train[:10000]")
    data = [{"en": item["translation"]["en"], "hi": item["translation"]["hi"]} for item in raw_dataset]
except Exception:
    raw_dataset = load_dataset("opus_books", "en-hi", split="train")
    data = [{"en": item["translation"]["en"], "hi": item["translation"]["hi"]} for item in raw_dataset]

df = pd.DataFrame(data)
df["en"] = df["en"].apply(lambda x: clean_sentence(x, is_hindi=False))
df["hi"] = df["hi"].apply(lambda x: clean_sentence(x, is_hindi=True))
df = df[(df["en"].str.len() > 0) & (df["hi"].str.len() > 0)].drop_duplicates()
print(f"Total cleaned parallel pairs: {len(df)}")
df.head()

In [ ]:
# 6. Train/Val/Test Split (80/10/10)
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

In [ ]:
# 7. PyTorch Dataset and Tokenization
class TranslationDataset(Dataset):
    def __init__(self, data_df, tokenizer, max_length=128):
        self.df = data_df
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        inputs = self.tokenizer(str(row["en"]), max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        labels = self.tokenizer(text_target=str(row["hi"]), max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")["input_ids"].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": labels
        }

model_name = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTForConditionalGeneration.from_pretrained(model_name).to(device)

batch_size = 16 if torch.cuda.is_available() else 4
train_loader = DataLoader(TranslationDataset(train_df, tokenizer), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TranslationDataset(val_df, tokenizer), batch_size=batch_size, shuffle=False)

In [ ]:
# 8. Optimizer, Scheduler, and Fine-Tuning Loop
epochs = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device)
            )
            total_val_loss += outputs.loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print("Saving best checkpoint to ./finetuned_en_hi...")
        model.save_pretrained("./finetuned_en_hi")
        tokenizer.save_pretrained("./finetuned_en_hi")

In [ ]:
# 9. Visualize Training and Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), history["train_loss"], marker="o", label="Training Loss")
plt.plot(range(1, epochs + 1), history["val_loss"], marker="s", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MarianMT Fine-Tuning Loss Curve (English → Hindi)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 10. Evaluation on Test Split with SacreBLEU
model.eval()
preds, refs = [], []

for _, row in test_df.head(100).iterrows():
    inputs = tokenizer(row["en"], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, num_beams=4, max_length=128)
    translation = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    preds.append(translation)
    refs.append(row["hi"])

bleu = sacrebleu.corpus_bleu(preds, [refs], tokenize="intl")
print(f"Test SacreBLEU Score: {bleu.score:.2f}")
print(f"Signature: {bleu}")

In [ ]:
# 11. Interactive Test Translation
sample_input = "Artificial intelligence is transforming the future of human communication."
inp = tokenizer(sample_input, return_tensors="pt").to(device)
with torch.no_grad():
    gen = model.generate(**inp, num_beams=4, max_length=128)
print("Source:", sample_input)
print("Translation:", tokenizer.decode(gen[0], skip_special_tokens=True))